### Import Dependencies

In [5]:
from google import genai
import instructor
from pydantic import BaseModel, Field
from qdrant_client import QdrantClient
from google.genai import types
import os

In [2]:
from dotenv import load_dotenv

load_dotenv("../../.env")

True

### Rag Pipeline

In [7]:
client = instructor.from_genai(
    gemini_client,
    model="gemini-3.1-flash-lite"
)

In [3]:
class RAGGenerationResponse(BaseModel):
    answer: str = Field(description="   answer to the question")

In [6]:
gemini_client = genai.Client(api_key=os.getenv("GOOGLE_API_KEY"))
qdrant_client = QdrantClient(url="http://localhost:6333")


def get_embeddings(text, task_type="SEMANTIC_SIMILARITY", model="gemini-embedding-001"):
    result = gemini_client.models.embed_content(
        model=model,
        contents=text,
        config=types.EmbedContentConfig(task_type=task_type)
    )
    return result.embeddings[0].values

def retrieve_data(query, k=5):
    query_embedding = get_embeddings(query)
    results = qdrant_client.query_points(
        collection_name="Amazon-items-collection-01",
        query=query_embedding,
        limit=k
    )
    retrieved_context_ids=[]
    retrieved_context=[]
    similarity_scores=[]
    retrieved_context_ratings=[]

    for result in results.points:
        retrieved_context_ids.append(result.payload["parent_asin"])
        retrieved_context.append(result.payload["preprocessed_description"])
        similarity_scores.append(result.score)
        retrieved_context_ratings.append(result.payload["average_rating"])

    return {
        "retrieved_context_ids": retrieved_context_ids,
        "retrieved_context": retrieved_context,
        "similarity_scores": similarity_scores,
        "retrieved_context_ratings": retrieved_context_ratings
    }


def process_context(context):
    formatted_context = ""
    for id, chunk, rating in zip(context["retrieved_context_ids"], context["retrieved_context"], context["retrieved_context_ratings"]):
        formatted_context += f"- ID: {id}, rating: {rating}, description: {chunk}\n"
    return formatted_context


def build_prompt(query, preprocessed_context):
    prompt = f"""
    You are a helpful assistant that can answer questions about the products in stock.
    You are given a question and a list of context.
    You need to answer the question based on the product descriptions.

    Instructions:
    - Answer the question based on the provided context only.
    - Never use the word "context" in your answer, refer it as the available products.
    - Do not use markdown formatting.

    Context:
    {preprocessed_context}

    Question: 
    {query}
    """
    return prompt

def generate_answer(prompt):
    response, RAW = client.create_with_completion(
    messages=[
        {"role": "user", "content": prompt}
    ],
    response_model=RAGGenerationResponse,
)

    return response

def rag_pipeline(query, topk_k=5):
    retrieved_context = retrieve_data(query, topk_k)
    preprocessed_context = process_context(retrieved_context)
    prompt = build_prompt(query, preprocessed_context)
    answer = generate_answer(prompt)

    final_answer = {
        "data_object": answer,
        "answer": answer.answer,
        "question": query,
        "retrieved_context_ids": retrieved_context["retrieved_context_ids"],
        "retrieved_context": retrieved_context["retrieved_context"]
    }
    return final_answer

In [8]:
output = rag_pipeline("Do you have any PC Items?")

In [9]:
output

{'data_object': RAGGenerationResponse(answer='Yes, based on the available products, we have the Klipsch ProMedia 2.0 Multimedia Compact Computer Speaker System, which is compatible with laptops, desktops, and gaming systems.'),
 'answer': 'Yes, based on the available products, we have the Klipsch ProMedia 2.0 Multimedia Compact Computer Speaker System, which is compatible with laptops, desktops, and gaming systems.',
 'question': 'Do you have any PC Items?',
 'retrieved_context_ids': ['B09XBFN4NR',
  'B0BG22K9GY',
  'B08DLWKXGL',
  'B0BM3CXBVD',
  'B0BK8CQPWD'],
 'retrieved_context': ['Klipsch ProMedia 2.0 Multimedia Compact Computer Speaker System Compatible for Any Laptop, Desktop, or Mobile Device for Premium Home Office, Workstation, or Gaming System Ultra Compact \xa0Bluetooth Streaming High-Performance Desktop System True High Fidelity - No Receiver Necessary Simple Set Up, Easy to Use Premium Look and Feel',
  'NIUTRENDZ Ultra Thin Case for Apple Pencil 2nd Generation Case Cover

### RAG Pipeline with grounding context

In [10]:
class RAGUsedContext(BaseModel):
    id: str = Field(description="   id of the item used to answer the question")
    description: str = Field(description="   description of the item used to answer the question")

class RAGGenerationResponse(BaseModel):
    answer: str = Field(description="   answer to the question")
    references: list[RAGUsedContext] = Field(description="   list of ritems used to answer the question")

In [11]:
gemini_client = genai.Client(api_key=os.getenv("GOOGLE_API_KEY"))
qdrant_client = QdrantClient(url="http://localhost:6333")


def get_embeddings(text, task_type="SEMANTIC_SIMILARITY", model="gemini-embedding-001"):
    result = gemini_client.models.embed_content(
        model=model,
        contents=text,
        config=types.EmbedContentConfig(task_type=task_type)
    )
    return result.embeddings[0].values

def retrieve_data(query, k=5):
    query_embedding = get_embeddings(query)
    results = qdrant_client.query_points(
        collection_name="Amazon-items-collection-01",
        query=query_embedding,
        limit=k
    )
    retrieved_context_ids=[]
    retrieved_context=[]
    similarity_scores=[]
    retrieved_context_ratings=[]

    for result in results.points:
        retrieved_context_ids.append(result.payload["parent_asin"])
        retrieved_context.append(result.payload["preprocessed_description"])
        similarity_scores.append(result.score)
        retrieved_context_ratings.append(result.payload["average_rating"])

    return {
        "retrieved_context_ids": retrieved_context_ids,
        "retrieved_context": retrieved_context,
        "similarity_scores": similarity_scores,
        "retrieved_context_ratings": retrieved_context_ratings
    }


def process_context(context):
    formatted_context = ""
    for id, chunk, rating in zip(context["retrieved_context_ids"], context["retrieved_context"], context["retrieved_context_ratings"]):
        formatted_context += f"- ID: {id}, rating: {rating}, description: {chunk}\n"
    return formatted_context


def build_prompt(query, preprocessed_context):
    prompt = f"""
    You are a helpful assistant that can answer questions about the products in stock.
    You are given a question and a list of context.
    You need to answer the question based on the product descriptions.

    Instructions:
    - Answer the question based on the provided context only.
    - Never use the word "context" in your answer, refer it as the available products.

    Context:
    {preprocessed_context}

    Question: 
    {query}
    """
    return prompt

def generate_answer(prompt):
    response, raw_response = client.create_with_completion(
    messages=[
        {"role": "user", "content": prompt}
    ],
    response_model=RAGGenerationResponse,
)

    return response

def rag_pipeline(query, topk_k=5):
    retrieved_context = retrieve_data(query, topk_k)
    preprocessed_context = process_context(retrieved_context)
    prompt = build_prompt(query, preprocessed_context)
    answer = generate_answer(prompt)

    final_answer = {
        "data_object": answer,
        "answer": answer.answer,
        "references": answer.references,
        "question": query,
        "retrieved_context_ids": retrieved_context["retrieved_context_ids"],
        "retrieved_context": retrieved_context["retrieved_context"]
    }
    return final_answer

In [12]:
output = rag_pipeline("Do you have any PC Items?")

In [13]:
output

{'data_object': RAGGenerationResponse(answer='Yes, among the available products, we have the Klipsch ProMedia 2.0 Multimedia Compact Computer Speaker System (ID: B09XBFN4NR), which is compatible with laptops and desktops. Additionally, the Anime Mixed Stickers (ID: B0BM3CXBVD) are suitable for personalizing computers.', references=[RAGUsedContext(id='B09XBFN4NR', description='Klipsch ProMedia 2.0 Multimedia Compact Computer Speaker System Compatible for Any Laptop, Desktop, or Mobile Device for Premium Home Office, Workstation, or Gaming System Ultra Compact \xa0Bluetooth Streaming High-Performance Desktop System True High Fidelity - No Receiver Necessary Simple Set Up, Easy to Use Premium Look and Feel'), RAGUsedContext(id='B0BM3CXBVD', description='Anime Mixed Stickers[100 Pcs] Vinyl Waterproof Stickers for Laptop Water Bottles for Hydro Flask Skateboard Computer Phone Anime Sticker Pack for Kids/Teen(Anime Mixed Stickers)')]),
 'answer': 'Yes, among the available products, we have t